# Résultats et Figures

Ce notebook présente de façon détaillée les deux parties du projet :
- Partie 1 : classification d'images avec Fashion-MNIST et CNN
- Partie 2 : analyse de sentiments avec IMDb et modèle d'embedding

Il contient :
1. Le code complet et détaillé de chaque partie.
2. Les étapes d'entraînement, d'évaluation et de sauvegarde.
3. Les résultats principaux (loss, accuracy, exemples d'erreurs).
4. Les commandes pour générer les figures et les modèles.

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from IPython.display import Image, display, Markdown

print('Répertoire courant :', os.getcwd())

# --- Partie 1 : Fashion-MNIST et CNN ---
print('\n=== Partie 1 : Classification d\'images ===')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

from models.model import FashionCNN
from utils.train import train_model
from utils.test import test_model

model = FashionCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

losses = train_model(model, train_loader, criterion, optimizer, epochs=5)
accuracy = test_model(model, test_loader)

print('Loss finale :', losses[-1])
print('Accuracy finale :', accuracy)

# Affichage de la courbe de loss si le fichier existe
if os.path.exists('loss_curve_fashion.png'):
    display(Markdown('### Courbe de loss Fashion-MNIST'))
    display(Image(filename='loss_curve_fashion.png'))
else:
    print('Fichier loss_curve_fashion.png introuvable')

# Affichage des erreurs si le fichier existe
if os.path.exists('erreurs_fashion.png'):
    display(Markdown('### Exemples d\'erreurs Fashion-MNIST'))
    display(Image(filename='erreurs_fashion.png'))
else:
    print('Fichier erreurs_fashion.png introuvable')


In [ ]:
# --- Partie 2 : Analyse de sentiments IMDb ---
print('\n=== Partie 2 : Analyse de sentiments ===')

from datasets import load_dataset
from collections import Counter
from models.sentiment_model import SentimentModel
from utils.dataset import SentimentDataset

raw = load_dataset('imdb')
train_texts = raw['train']['text']
train_labels = raw['train']['label']
test_texts = raw['test']['text']
test_labels = raw['test']['label']

counter = Counter()
for text in train_texts:
    counter.update(text.lower().split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common(10000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab) + 2

train_dataset = SentimentDataset(train_texts, train_labels, vocab)
test_dataset = SentimentDataset(test_texts, test_labels, vocab)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

model_sent = SentimentModel(vocab_size=vocab_size, emb_dim=64)
criterion_sent = nn.BCEWithLogitsLoss()
optimizer_sent = optim.Adam(model_sent.parameters(), lr=0.001)

losses_sent = []
for epoch in range(5):
    model_sent.train()
    running_loss = 0.0
    for texts, labels in train_loader:
        optimizer_sent.zero_grad()
        outputs = model_sent(texts).squeeze(1)
        loss = criterion_sent(outputs, labels)
        loss.backward()
        optimizer_sent.step()
        running_loss += loss.item()
    losses_sent.append(running_loss)
    print(f'Epoch {epoch+1}, Loss: {running_loss:.4f}')

print('Loss finale sentiment :', losses_sent[-1])

if os.path.exists('loss_sentiment.png'):
    display(Markdown('### Courbe de loss Sentiment IMDb'))
    display(Image(filename='loss_sentiment.png'))
else:
    print('Fichier loss_sentiment.png introuvable')
